In [ ]:
%matplotlib ipympl

In [34]:
import mlflow
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import linregress

from benchmark.config import EXPERIMENT_NAME


In [35]:
exp = mlflow.set_experiment("event_categories")

In [36]:
# one row per child run (a model x system-prompt pair) with each scorer's
# mean over the run; the model is only recorded in the run name. The
# per-trace "<scorer>/by_price_time" series only show their last point in a
# runs table, so they are left out.
run_name = "sweep-20261101-2146"
# the parentRunId tag holds the parent's run_id, not its name, so look the
# parent up by name first
parent = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string=f"tags.mlflow.runName = '{run_name}'",
)
# assert len(parent) == 1, f"expected one run named {run_name!r}, found {len(parent)}"
runs = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string=f"tags.mlflow.parentRunId = '{parent.run_id.iloc[0]}'",
)

In [37]:
# every trace (one per predict call) of every child run, tagged with the run
# it came from. The record's inputs sit under request["inputs"], the market
# category is a tag copied onto the trace, and each scorer's value is an
# assessment on the trace.
traces = pd.concat(
    [
        mlflow.search_traces(locations=[exp.experiment_id], run_id=run_id).assign(
            run=name
        )
        for run_id, name in zip(runs["run_id"], runs["tags.mlflow.runName"])
    ],
    ignore_index=True,
)

In [38]:
def assessment_value(assessments: list, name: str) -> float | None:
    # scorer outputs are stored as "feedback", dataset expectations (the
    # market's own probability) as "expectation"
    for a in assessments:
        if a["assessment_name"] == name:
            return (a.get("feedback") or a["expectation"])["value"]
    return None


In [39]:
trace_df = pd.DataFrame(
    {
        "run": traces["run"],
        "model": traces["request"].map(lambda r: r["model"]),
        "system_prompt": traces["request"].map(lambda r: r["system_prompt"]),
        # a tag on the child run rather than on the trace, as "YYYY-MM"; taken
        # as the last day of that month so it compares against as_of
        "training_data_cutoff": pd.PeriodIndex(
            traces["run"].map(
                dict(
                    zip(runs["tags.mlflow.runName"], runs["tags.training_data_cutoff"])
                )
            ),
            freq="M",
        )
        .to_timestamp(how="end")
        .normalize(),
        "exchange": traces["tags"].map(lambda t: t.get("exchange")),
        "category": traces["tags"].map(lambda t: t.get("event_category")),
        "event_type": traces["tags"].map(lambda t: t.get("std_event_type")),
        "market_title": traces["request"].map(lambda r: r["inputs"]["market_title"]),
        "response": traces["response"].map(lambda r: r and r["raw"]),
        "as_of": traces["request"].map(lambda r: pd.Timestamp(r["inputs"]["as_of"])),
        "predicted_probability": traces["response"].map(
            lambda r: r and r["probability"]
        ),
        "market_probability": traces["assessments"].map(
            lambda a: assessment_value(a, "market_probability")
        ),
        "state": traces["state"],
    }
)
trace_df = trace_df[trace_df["system_prompt"] == "no-cutoff"]
trace_df = trace_df[trace_df["as_of"] > pd.Timestamp("2026-06-30")]

In [40]:
trace_df["error"] = abs(
    trace_df["market_probability"] - trace_df["predicted_probability"]
)

In [41]:
trace_df[
    [
        "event_type",
        "market_title",
        "response",
        "as_of",
        "error",
        "predicted_probability",
        "market_probability",
    ]
]

,event_type,market_title,response,as_of,error,predicted_probability,market_probability
0,corporate KPI,Will Piper Sandler (PIPR) beat quarterly earni...,Piper Sandler has a mixed record against conse...,2026-07-19 00:00:16,0.180,0.680,0.500
5,IPO,"Will Bloomberg IPO by March 31, 2027?",Bloomberg has long been privately held by Mich...,2026-07-31 00:00:20,0.052,0.003,0.055
7,None,Will Lam Research (LRCX) Q1 total revenue be a...,I couldn't check Lam's latest guidance or cons...,2026-10-13 00:00:29,0.350,0.570,0.220
9,IPO,Will Anthropic's initial public offering occur...,I put the chance that Anthropic's IPO lands in...,2026-08-20 00:00:15,0.015,0.040,0.055
15,valuation,Will ByteDance's valuation hit (HIGH) $700B by...,"I couldn't check NPM data directly, so this is...",2026-07-02 00:00:04,0.310,0.100,0.410
16,None,Will the margin of victory for Dan Osborn in t...,"I didn't look up current polling, so this esti...",2026-10-28 04:00:00,0.040,0.140,0.180
18,executive order,Will Donald Trump issue any executive action o...,Comprehensive trade embargoes on a new country...,2026-07-12 04:00:00,0.090,0.040,0.130
20,speech quote,"Will Trump say ""Epstein"" this week?","Trump says ""Epstein"" very frequently when aske...",2026-07-21 00:00:06,0.820,0.880,0.060
22,election,Will Russian Party – Serbia in BRICS – Sloboda...,The Russian Party list led by Slobodan Nikolić...,2026-09-22 00:00:13,0.133,0.002,0.135
27,None,How many initial jobless claims will there be ...,Recent initial claims in 2025–2026 have mostly...,2026-10-30 04:00:00,0.060,0.950,0.890


In [ ]:
trace_df["error"].hist(bins=20, figsize=(8, 4))

In [ ]:
# months from the model's training cutoff to the market price time, positive
# when the market was priced after the cutoff (fractional: days / mean month)
trace_df["time_diff"] = (
    trace_df["as_of"] - trace_df["training_data_cutoff"]
).dt.days / (365.25 / 12)


In [ ]:
# one series per model so the legend says which model each point is
ax = None
for i, (model, points) in enumerate(trace_df.groupby("model")):
    ax = points.plot.scatter(
        x="time_diff", y="error", label=model, color=f"C{i}", ax=ax
    )
ax.set_xlabel("months after training cutoff")

In [ ]:
# rows whose forecast could not be parsed have no error and would make the
# fit NaN
scored = trace_df.dropna(subset=["error"])


In [ ]:
fit = smf.ols("error ~ time_diff + C(model)", data=scored).fit()


In [ ]:
fit.summary()


In [ ]:
# the same fit per model, slope in error per month with its p-value
by_model = scored.groupby("model").apply(
    lambda g: pd.Series(linregress(g["time_diff"], g["error"])._asdict()),
    include_groups=False,
)


In [ ]:
# the fitted line per model: common slope, model-specific intercept
for i, (model, points) in enumerate(scored.groupby("model")):
    x = points["time_diff"].sort_values()
    ax.plot(
        x,
        fit.predict(pd.DataFrame({"time_diff": x, "model": model})),
        color=f"C{i}",
    )
ax.set_title(
    f"error vs months after cutoff; slope {fit.params['time_diff']:.4f}/month (p={fit.pvalues['time_diff']:.3f})"
)
ax.figure
